# SigFlow-Sim — No-`yfinance` Runnable Version

This notebook predicts the probability distribution of future realised volatility and classifies the forecast period as a low-, medium-, or high-volatility regime.

This version is designed to run in restricted Python environments:

- **no `pip install` step**;
- **no `yfinance`, `iisignature`, or `nflows` dependency**;
- real prices are requested directly from Yahoo Finance's chart endpoint;
- Stooq CSV data is used as a second real-data provider;
- a clearly labelled synthetic fallback is used only if both internet sources fail;
- every major stage prints progress;
- the signature and conditional normalising flow are implemented directly with NumPy and PyTorch.

## Running it

Use **Kernel → Restart and Run All** in Jupyter, or **Runtime → Run all** in Google Colab.

Keep `QUICK_MODE = True` for the first successful run. Afterwards, set it to `False` for the larger experiment.


In [ ]:
# Environment diagnostic — no package installation is attempted.

import importlib.util
import sys

REQUIRED_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "torch": "torch",
}

missing = [
    package
    for package, import_name in REQUIRED_IMPORTS.items()
    if importlib.util.find_spec(import_name) is None
]

print("Python executable:", sys.executable)
if missing:
    raise ModuleNotFoundError(
        "This notebook requires the environment's core scientific packages: "
        + ", ".join(missing)
    )

print("Core packages are available.")
print("No yfinance installation is required.")


In [ ]:
from __future__ import annotations

import copy
import io
import json
import math
import random
import urllib.error
import urllib.parse
import urllib.request
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPS = 1e-8
REGIME_NAMES = np.array(["Low", "Medium", "High"])

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)
print("Market-data route: direct Yahoo endpoint → Stooq CSV → synthetic fallback")


## Configuration

`QUICK_MODE=True` is a small but real experiment intended to prove that the complete pipeline works. It uses fewer tickers, fewer dates, fewer epochs, and fewer predictive samples.

Set `QUICK_MODE=False` only after the quick run succeeds.

In [ ]:
QUICK_MODE = True

@dataclass
class Config:
    quick_mode: bool = QUICK_MODE

    # Real market data is attempted first. If unavailable, synthetic data is
    # used and prominently identified in the output.
    use_real_market_data: bool = True
    allow_synthetic_fallback: bool = True
    refresh_data: bool = False

    tickers: tuple[str, ...] = (
        ("AAPL", "MSFT") if QUICK_MODE
        else ("AAPL", "MSFT", "GOOGL", "AMZN")
    )
    start_date: str = "2021-01-01" if QUICK_MODE else "2016-01-01"
    end_date: str = "2026-07-25"
    cache_dir: str = "sigflow_cache"
    output_dir: str = "sigflow_working_outputs"

    window: int = 20
    horizon: int = 10
    annualisation: float = 252.0
    ewma_lambda: float = 0.94
    signature_depth: int = 3

    train_fraction: float = 0.70
    validation_fraction: float = 0.15

    regimes: int = 3
    hidden_size: int = 48 if QUICK_MODE else 96
    dropout: float = 0.10

    expert_alignment_weight: float = 0.35
    regime_classification_weight: float = 0.25
    label_smoothing: float = 0.02

    epochs: int = 12 if QUICK_MODE else 100
    batch_size: int = 128
    learning_rate: float = 2e-4
    weight_decay: float = 1e-5
    gradient_clip: float = 1.0
    patience: int = 5 if QUICK_MODE else 15
    print_every: int = 1 if QUICK_MODE else 5

    prediction_samples: int = 128 if QUICK_MODE else 512
    prediction_batch_size: int = 256
    seed: int = 42

CFG = Config()
print(json.dumps(asdict(CFG), indent=2))

## Reproducibility, scaling, and data containers

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class Standardiser:
    mean: np.ndarray
    scale: np.ndarray

    @classmethod
    def fit(cls, values: np.ndarray) -> "Standardiser":
        mean = np.nanmean(values, axis=0)
        scale = np.nanstd(values, axis=0)
        scale = np.where(np.isfinite(scale) & (scale > 1e-6), scale, 1.0)
        return cls(mean.astype(np.float32), scale.astype(np.float32))

    def transform(self, values: np.ndarray) -> np.ndarray:
        transformed = (values - self.mean) / self.scale
        return np.nan_to_num(
            transformed, nan=0.0, posinf=0.0, neginf=0.0
        ).astype(np.float32)


@dataclass
class MarketDataset:
    features: np.ndarray
    targets_log_vol: np.ndarray
    metadata: pd.DataFrame
    feature_names: list[str]
    data_source: str


@dataclass
class PreparedSplit:
    train_indices: np.ndarray
    validation_indices: np.ndarray
    test_indices: np.ndarray
    context: np.ndarray
    regime_labels: np.ndarray
    regime_thresholds: np.ndarray
    standardiser: Standardiser
    train_cutoff: pd.Timestamp
    validation_cutoff: pd.Timestamp


set_seed(CFG.seed)

## Dependency-free path signature

For each piecewise-linear path increment \(a\), the straight-segment signature is:

- level 1: \(a\)
- level 2: \(a \otimes a / 2\)
- level 3: \(a \otimes a \otimes a / 6\)

Chen's identity combines successive segments. This gives an ordinary truncated signature without requiring the compiled `iisignature` package.

In [ ]:
def truncated_signature(path: np.ndarray, depth: int = 3) -> np.ndarray:
    if depth not in (1, 2, 3):
        raise ValueError("This reliable implementation supports depths 1, 2, or 3.")

    increments = np.diff(np.asarray(path, dtype=np.float64), axis=0)
    dimension = path.shape[1]

    level1 = np.zeros(dimension, dtype=np.float64)
    level2 = np.zeros((dimension, dimension), dtype=np.float64)
    level3 = np.zeros((dimension, dimension, dimension), dtype=np.float64)

    for increment in increments:
        segment1 = increment
        segment2 = np.einsum("i,j->ij", increment, increment) / 2.0
        segment3 = (
            np.einsum("i,j,k->ijk", increment, increment, increment) / 6.0
        )

        old_level1 = level1.copy()
        old_level2 = level2.copy()

        level1 = old_level1 + segment1
        if depth >= 2:
            level2 = (
                old_level2
                + np.einsum("i,j->ij", old_level1, segment1)
                + segment2
            )
        if depth >= 3:
            level3 = (
                level3
                + np.einsum("ij,k->ijk", old_level2, segment1)
                + np.einsum("i,jk->ijk", old_level1, segment2)
                + segment3
            )

    levels = [level1.ravel()]
    if depth >= 2:
        levels.append(level2.ravel())
    if depth >= 3:
        levels.append(level3.ravel())
    return np.concatenate(levels)


def signature_feature_names(path_dimension: int, depth: int) -> list[str]:
    names: list[str] = []
    for level in range(1, depth + 1):
        for index in range(path_dimension**level):
            names.append(f"signature_L{level}_{index}")
    return names

## Market-data loading without `yfinance`

The notebook uses two real-data routes that require no extra Python package:

1. Yahoo Finance's chart response;
2. Stooq's daily CSV response.

The cache is checked before either request. If the runtime blocks all external internet traffic, the notebook can still exercise the complete modelling pipeline using a prominently labelled synthetic fallback.


In [ ]:
def synthetic_close_series(
    ticker: str,
    start_date: str,
    end_date: str,
    seed: int,
) -> pd.Series:
    dates = pd.date_range(start_date, end_date, freq="B", inclusive="left")
    rng = np.random.default_rng(seed)

    regimes = np.zeros(len(dates), dtype=int)
    transition = np.array([
        [0.975, 0.023, 0.002],
        [0.030, 0.940, 0.030],
        [0.010, 0.080, 0.910],
    ])
    for i in range(1, len(dates)):
        regimes[i] = rng.choice(3, p=transition[regimes[i - 1]])

    daily_volatility = np.array([0.008, 0.015, 0.030])[regimes]
    returns = rng.normal(0.00025, daily_volatility)
    prices = 100.0 * np.exp(np.cumsum(returns))
    return pd.Series(prices, index=dates, name="Close")


def clean_close_series(close: pd.Series, ticker: str) -> pd.Series:
    close = pd.Series(close, dtype=float, name="Close")
    close.index = pd.to_datetime(close.index, errors="coerce")
    if getattr(close.index, "tz", None) is not None:
        close.index = close.index.tz_localize(None)

    close = close[~close.index.isna()]
    close = close.replace([np.inf, -np.inf], np.nan).dropna()
    close = close[close > 0]
    close = close[~close.index.duplicated(keep="last")].sort_index()

    if close.empty:
        raise ValueError(f"No usable closing prices were returned for {ticker}.")
    return close


def request_bytes(url: str, timeout: int = 30) -> bytes:
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 Chrome/124 Safari/537.36"
            ),
            "Accept": "application/json,text/csv,*/*",
        },
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def download_yahoo_chart(
    ticker: str,
    start_date: str,
    end_date: str,
) -> pd.Series:
    period1 = int(pd.Timestamp(start_date, tz="UTC").timestamp())
    period2 = int(pd.Timestamp(end_date, tz="UTC").timestamp())

    parameters = urllib.parse.urlencode({
        "period1": period1,
        "period2": period2,
        "interval": "1d",
        "events": "history",
        "includeAdjustedClose": "true",
    })
    quoted_ticker = urllib.parse.quote(ticker, safe="")
    url = (
        f"https://query1.finance.yahoo.com/v8/finance/chart/"
        f"{quoted_ticker}?{parameters}"
    )

    payload = json.loads(request_bytes(url).decode("utf-8"))
    chart = payload.get("chart", {})
    if chart.get("error"):
        raise RuntimeError(str(chart["error"]))

    results = chart.get("result")
    if not results:
        raise ValueError(f"Yahoo returned no chart result for {ticker}.")

    result = results[0]
    timestamps = result.get("timestamp") or []
    indicators = result.get("indicators") or {}

    adjusted_blocks = indicators.get("adjclose") or []
    adjusted = adjusted_blocks[0].get("adjclose") if adjusted_blocks else None

    if adjusted is None:
        quote_blocks = indicators.get("quote") or []
        adjusted = quote_blocks[0].get("close") if quote_blocks else None

    if adjusted is None or len(timestamps) != len(adjusted):
        raise ValueError(f"Yahoo returned incomplete prices for {ticker}.")

    dates = pd.to_datetime(timestamps, unit="s", utc=True).tz_localize(None)
    return clean_close_series(
        pd.Series(adjusted, index=dates, name="Close"),
        ticker,
    )


def stooq_symbol(ticker: str) -> str:
    # These experiments use US-listed equities. Stooq represents them with
    # a lower-case ".us" suffix.
    return ticker.lower().replace(".", "-") + ".us"


def download_stooq_csv(
    ticker: str,
    start_date: str,
    end_date: str,
) -> pd.Series:
    parameters = urllib.parse.urlencode({
        "s": stooq_symbol(ticker),
        "d1": pd.Timestamp(start_date).strftime("%Y%m%d"),
        "d2": pd.Timestamp(end_date).strftime("%Y%m%d"),
        "i": "d",
    })
    url = f"https://stooq.com/q/d/l/?{parameters}"
    raw = request_bytes(url)

    frame = pd.read_csv(io.BytesIO(raw))
    required = {"Date", "Close"}
    if frame.empty or not required.issubset(frame.columns):
        preview = raw[:160].decode("utf-8", errors="replace")
        raise ValueError(
            f"Stooq returned no usable CSV for {ticker}. Response: {preview!r}"
        )

    return clean_close_series(
        pd.Series(
            pd.to_numeric(frame["Close"], errors="coerce").to_numpy(),
            index=pd.to_datetime(frame["Date"], errors="coerce"),
            name="Close",
        ),
        ticker,
    )


def cached_close_series(ticker: str, cfg: Config) -> pd.Series | None:
    cache_dir = Path(cfg.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{ticker}_{cfg.start_date}_{cfg.end_date}.csv"

    if not cache_path.exists() or cfg.refresh_data:
        return None

    cached = pd.read_csv(cache_path, index_col=0, parse_dates=True)
    if "Close" not in cached.columns or cached.empty:
        return None

    print(f"  {ticker}: loaded cached prices")
    return clean_close_series(cached["Close"], ticker)


def save_price_cache(ticker: str, close: pd.Series, cfg: Config) -> None:
    cache_dir = Path(cfg.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{ticker}_{cfg.start_date}_{cfg.end_date}.csv"
    close.to_frame().to_csv(cache_path)


def download_close_series(
    ticker: str,
    cfg: Config,
) -> tuple[pd.Series, str]:
    cached = cached_close_series(ticker, cfg)
    if cached is not None:
        return cached, "cache"

    errors: list[str] = []

    for provider_name, downloader in [
        ("yahoo_direct", download_yahoo_chart),
        ("stooq_direct", download_stooq_csv),
    ]:
        try:
            close = downloader(ticker, cfg.start_date, cfg.end_date)
            save_price_cache(ticker, close, cfg)
            print(
                f"  {ticker}: downloaded {len(close):,} prices "
                f"through {provider_name}"
            )
            return close, provider_name
        except Exception as exc:
            message = f"{provider_name}: {type(exc).__name__}: {exc}"
            errors.append(message)
            print(f"  {ticker}: {message}")

    raise RuntimeError(
        f"All direct real-data providers failed for {ticker}. "
        + " | ".join(errors)
    )


def load_all_prices(cfg: Config) -> tuple[dict[str, pd.Series], str]:
    prices: dict[str, pd.Series] = {}
    providers: list[str] = []

    if cfg.use_real_market_data:
        try:
            print("Attempting direct real market-data download...")
            for ticker in cfg.tickers:
                prices[ticker], provider = download_close_series(ticker, cfg)
                providers.append(provider)

            unique_providers = "+".join(sorted(set(providers)))
            return prices, f"real_{unique_providers}"
        except Exception as exc:
            print("\nReal market-data loading failed:")
            print(type(exc).__name__ + ":", exc)

            if not cfg.allow_synthetic_fallback:
                raise

            print(
                "\nWARNING: continuing with synthetic data because this runtime "
                "could not reach either real-data provider. The resulting metrics "
                "test the pipeline only; they are not evidence about real stocks."
            )

    for ticker_id, ticker in enumerate(cfg.tickers):
        prices[ticker] = synthetic_close_series(
            ticker,
            cfg.start_date,
            cfg.end_date,
            cfg.seed + 1000 * ticker_id,
        )
    return prices, "synthetic_fallback"


## Leakage-safe features and targets

In [ ]:
STATISTIC_NAMES = [
    "annualised_mean_return",
    "log_annualised_volatility",
    "mean_absolute_return",
    "downside_annualised_volatility",
    "upside_annualised_volatility",
    "maximum_absolute_return",
    "last_return",
    "cumulative_return",
    "lag1_autocorrelation",
    "skewness",
    "excess_kurtosis",
    "volatility_of_volatility",
    "squared_return_trend",
]


def safe_autocorrelation(values: np.ndarray) -> float:
    if len(values) < 3:
        return 0.0
    left, right = values[:-1], values[1:]
    if np.std(left) < EPS or np.std(right) < EPS:
        return 0.0
    value = np.corrcoef(left, right)[0, 1]
    return float(value) if np.isfinite(value) else 0.0


def volatility_of_volatility(values: np.ndarray, subwindow: int = 5) -> float:
    if len(values) < subwindow:
        return 0.0
    rolling_ms = np.convolve(
        values**2,
        np.ones(subwindow) / subwindow,
        mode="valid",
    )
    return float(np.std(np.sqrt(np.maximum(rolling_ms, 0.0))))


def statistical_features(past: np.ndarray, annualisation: float) -> np.ndarray:
    mean_return = float(np.mean(past))
    sample_std = max(float(np.std(past, ddof=1)), EPS)
    annualised_vol = math.sqrt(annualisation) * sample_std

    downside = np.minimum(past, 0.0)
    upside = np.maximum(past, 0.0)
    z = (past - mean_return) / sample_std

    time = np.arange(len(past), dtype=float)
    centred_time = time - time.mean()
    denominator = max(float(np.sum(centred_time**2)), EPS)
    squared_trend = float(
        np.sum(centred_time * (past**2 - np.mean(past**2))) / denominator
    )

    return np.array([
        annualisation * mean_return,
        math.log(annualised_vol + EPS),
        np.mean(np.abs(past)),
        math.sqrt(annualisation * np.mean(downside**2)),
        math.sqrt(annualisation * np.mean(upside**2)),
        np.max(np.abs(past)),
        past[-1],
        np.sum(past),
        safe_autocorrelation(past),
        np.mean(z**3),
        np.mean(z**4) - 3.0,
        volatility_of_volatility(past),
        squared_trend,
    ], dtype=np.float64)


def make_signature_path(past: np.ndarray) -> np.ndarray:
    mean = np.mean(past)
    scale = max(float(np.std(past, ddof=1)), EPS)
    standardised = (past - mean) / scale

    time = np.linspace(0.0, 1.0, len(past) + 1)
    cumulative = np.concatenate(
        [[0.0], np.cumsum(standardised) / math.sqrt(len(past))]
    )
    quadratic_variation = np.concatenate(
        [[0.0], np.cumsum(standardised**2) / len(past)]
    )
    return np.column_stack([time, cumulative, quadratic_variation])


def realised_volatility(values: np.ndarray, annualisation: float) -> float:
    return float(np.sqrt(annualisation * np.mean(values**2)))


def ewma_volatility(values: np.ndarray, annualisation: float, decay: float) -> float:
    variance = max(float(np.var(values, ddof=1)), EPS)
    for value in values:
        variance = decay * variance + (1.0 - decay) * float(value**2)
    return math.sqrt(annualisation * max(variance, EPS))


def build_market_dataset(cfg: Config) -> MarketDataset:
    prices, data_source = load_all_prices(cfg)
    feature_rows: list[np.ndarray] = []
    target_rows: list[float] = []
    records: list[dict[str, object]] = []

    signature_names = signature_feature_names(3, cfg.signature_depth)
    feature_names = signature_names + STATISTIC_NAMES

    print("\nBuilding ticker-safe samples...")
    for ticker_id, ticker in enumerate(cfg.tickers):
        close = prices[ticker].dropna().sort_index()
        log_returns = np.log(close).diff().dropna()
        values = log_returns.to_numpy(dtype=np.float64)
        dates = pd.DatetimeIndex(log_returns.index)

        if len(values) < cfg.window + cfg.horizon:
            raise ValueError(f"Not enough observations for {ticker}.")

        ticker_count = 0
        for i in range(cfg.window, len(values) - cfg.horizon + 1):
            past = values[i - cfg.window:i]
            future = values[i:i + cfg.horizon]

            path = make_signature_path(past)
            signature = truncated_signature(path, cfg.signature_depth)
            statistics = statistical_features(past, cfg.annualisation)
            features = np.concatenate([signature, statistics])

            actual_vol = realised_volatility(future, cfg.annualisation)
            feature_rows.append(features)
            target_rows.append(math.log(actual_vol + EPS))
            records.append({
                "ticker_id": ticker_id,
                "ticker": ticker,
                "origin_date": dates[i - 1],
                "target_start_date": dates[i],
                "target_end_date": dates[i + cfg.horizon - 1],
                "actual_vol": actual_vol,
                "rolling_vol_baseline": realised_volatility(
                    past, cfg.annualisation
                ),
                "ewma_vol_baseline": ewma_volatility(
                    past, cfg.annualisation, cfg.ewma_lambda
                ),
                "data_source": data_source,
            })
            ticker_count += 1
        print(f"  {ticker}: {ticker_count:,} samples")

    original_metadata = pd.DataFrame(records)
    ordering = (
        original_metadata.assign(_position=np.arange(len(original_metadata)))
        .sort_values(["origin_date", "ticker"])["_position"]
        .to_numpy(dtype=int)
    )

    metadata = original_metadata.iloc[ordering].reset_index(drop=True)
    features = np.asarray(feature_rows, dtype=np.float32)[ordering]
    targets = np.asarray(target_rows, dtype=np.float32)[ordering]

    if not np.all(np.isfinite(features)):
        raise ValueError("Non-finite feature values remain.")
    if not np.all(np.isfinite(targets)):
        raise ValueError("Non-finite target values remain.")

    print(
        f"Dataset complete: {len(metadata):,} samples, "
        f"{features.shape[1]} numerical features, source={data_source}"
    )
    return MarketDataset(
        features=features,
        targets_log_vol=targets,
        metadata=metadata,
        feature_names=feature_names,
        data_source=data_source,
    )

## Purged chronological split and training-only scaling

In [ ]:
def prepare_split(dataset: MarketDataset, cfg: Config) -> PreparedSplit:
    metadata = dataset.metadata
    unique_dates = np.array(sorted(pd.to_datetime(metadata["origin_date"]).unique()))

    train_position = int(len(unique_dates) * cfg.train_fraction) - 1
    validation_position = int(
        len(unique_dates) * (cfg.train_fraction + cfg.validation_fraction)
    ) - 1
    train_position = max(0, min(train_position, len(unique_dates) - 3))
    validation_position = max(
        train_position + 1,
        min(validation_position, len(unique_dates) - 2),
    )

    train_cutoff = pd.Timestamp(unique_dates[train_position])
    validation_cutoff = pd.Timestamp(unique_dates[validation_position])

    origin = pd.to_datetime(metadata["origin_date"])
    target_end = pd.to_datetime(metadata["target_end_date"])

    train_indices = np.flatnonzero((target_end <= train_cutoff).to_numpy())
    validation_indices = np.flatnonzero(
        ((origin > train_cutoff) & (target_end <= validation_cutoff)).to_numpy()
    )
    test_indices = np.flatnonzero((origin > validation_cutoff).to_numpy())

    if min(len(train_indices), len(validation_indices), len(test_indices)) == 0:
        raise ValueError("A chronological split is empty.")

    standardiser = Standardiser.fit(dataset.features[train_indices])
    scaled = standardiser.transform(dataset.features)
    ticker_one_hot = np.eye(len(cfg.tickers), dtype=np.float32)[
        metadata["ticker_id"].to_numpy(dtype=int)
    ]
    context = np.concatenate([scaled, ticker_one_hot], axis=1).astype(np.float32)

    thresholds = np.quantile(
        dataset.targets_log_vol[train_indices],
        [1 / 3, 2 / 3],
    )
    labels = np.digitize(dataset.targets_log_vol, thresholds).astype(np.int64)

    metadata.loc[:, "split"] = "unused"
    metadata.loc[train_indices, "split"] = "train"
    metadata.loc[validation_indices, "split"] = "validation"
    metadata.loc[test_indices, "split"] = "test"
    metadata.loc[:, "regime"] = labels
    metadata.loc[:, "regime_name"] = REGIME_NAMES[labels]

    print("\nPurged chronological split:")
    print(f"  Train:      {len(train_indices):,}")
    print(f"  Validation: {len(validation_indices):,}")
    print(f"  Test:       {len(test_indices):,}")
    print(f"  Train cutoff:      {train_cutoff.date()}")
    print(f"  Validation cutoff: {validation_cutoff.date()}")
    print(
        "  Training regime thresholds: "
        f"{math.exp(thresholds[0]):.3f}, {math.exp(thresholds[1]):.3f}"
    )

    return PreparedSplit(
        train_indices=train_indices,
        validation_indices=validation_indices,
        test_indices=test_indices,
        context=context,
        regime_labels=labels,
        regime_thresholds=thresholds.astype(np.float32),
        standardiser=standardiser,
        train_cutoff=train_cutoff,
        validation_cutoff=validation_cutoff,
    )

## Conditional flow mixture

Each expert uses an invertible **sinh–arcsinh flow**. A standard-normal variable is transformed into a context-dependent distribution with its own:

- location;
- scale;
- skewness;
- tail weight.

The gate assigns probabilities to the low-, medium-, and high-volatility experts.

In [ ]:
def stable_log_cosh(value: torch.Tensor) -> torch.Tensor:
    return torch.logaddexp(value, -value) - math.log(2.0)


class ConditionalSinhArcsinhExpert(nn.Module):
    def __init__(self, context_dimension: int, hidden: int, dropout: float):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(context_dimension, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 4),
        )
        final_layer = self.network[-1]
        nn.init.zeros_(final_layer.weight)
        with torch.no_grad():
            final_layer.bias[:] = torch.tensor([-1.8, -0.6, 0.0, 0.0])

    def parameters_from_context(
        self, context: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        raw = self.network(context)
        location = raw[:, 0]
        scale = F.softplus(raw[:, 1]) + 0.05
        skew = 2.0 * torch.tanh(raw[:, 2])
        tail = 0.55 + 1.45 * torch.sigmoid(raw[:, 3])
        return location, scale, skew, tail

    def log_prob(self, target: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        target = target.squeeze(-1)
        location, scale, skew, tail = self.parameters_from_context(context)

        standardised = (target - location) / scale
        inverse_argument = tail * torch.asinh(standardised) - skew
        inverse_argument = torch.clamp(inverse_argument, -12.0, 12.0)
        base = torch.sinh(inverse_argument)

        log_base_density = -0.5 * base**2 - 0.5 * math.log(2.0 * math.pi)
        log_abs_jacobian = (
            torch.log(tail)
            - torch.log(scale)
            + stable_log_cosh(inverse_argument)
            - 0.5 * torch.log1p(standardised**2)
        )
        return log_base_density + log_abs_jacobian

    def sample(self, number_of_samples: int, context: torch.Tensor) -> torch.Tensor:
        location, scale, skew, tail = self.parameters_from_context(context)
        noise = torch.randn(
            context.shape[0],
            number_of_samples,
            device=context.device,
            dtype=context.dtype,
        )
        transformed = torch.sinh(
            (torch.asinh(noise) + skew.unsqueeze(1)) / tail.unsqueeze(1)
        )
        return location.unsqueeze(1) + scale.unsqueeze(1) * transformed


class RegimeFlowMixture(nn.Module):
    def __init__(self, context_dimension: int, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.gate = nn.Sequential(
            nn.Linear(context_dimension, cfg.hidden_size),
            nn.LayerNorm(cfg.hidden_size),
            nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.hidden_size, cfg.hidden_size),
            nn.SiLU(),
            nn.Linear(cfg.hidden_size, cfg.regimes),
        )
        self.experts = nn.ModuleList([
            ConditionalSinhArcsinhExpert(
                context_dimension,
                cfg.hidden_size,
                cfg.dropout,
            )
            for _ in range(cfg.regimes)
        ])

    def component_log_probabilities(
        self,
        context: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:
        return torch.stack(
            [expert.log_prob(target, context) for expert in self.experts],
            dim=1,
        )

    def mixture_outputs(
        self,
        context: torch.Tensor,
        target: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        logits = self.gate(context)
        component_log_probabilities = self.component_log_probabilities(
            context, target
        )
        mixture_log_probability = torch.logsumexp(
            F.log_softmax(logits, dim=1) + component_log_probabilities,
            dim=1,
        )
        return mixture_log_probability, logits, component_log_probabilities

    def sample_mixture(
        self,
        context: torch.Tensor,
        number_of_samples: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        logits = self.gate(context)
        probabilities = F.softmax(logits, dim=1)

        component_samples = torch.stack(
            [
                expert.sample(number_of_samples, context)
                for expert in self.experts
            ],
            dim=1,
        )  # [batch, regime, samples]

        selected_components = torch.distributions.Categorical(
            probs=probabilities
        ).sample((number_of_samples,)).transpose(0, 1)

        selected = torch.gather(
            component_samples.permute(0, 2, 1),
            dim=2,
            index=selected_components.unsqueeze(-1),
        ).squeeze(-1)
        return selected, probabilities

## Training with validation, early stopping, and clipping

In [ ]:
def make_loader(
    split: PreparedSplit,
    dataset: MarketDataset,
    indices: np.ndarray,
    cfg: Config,
    shuffle: bool,
) -> DataLoader:
    tensor_dataset = TensorDataset(
        torch.from_numpy(split.context[indices]).float(),
        torch.from_numpy(dataset.targets_log_vol[indices]).float().unsqueeze(1),
        torch.from_numpy(split.regime_labels[indices]).long(),
    )
    return DataLoader(
        tensor_dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        drop_last=False,
    )


def objective(
    model: RegimeFlowMixture,
    context: torch.Tensor,
    target: torch.Tensor,
    labels: torch.Tensor,
    cfg: Config,
) -> tuple[torch.Tensor, dict[str, float]]:
    mixture_log_probability, logits, component_log_probabilities = (
        model.mixture_outputs(context, target)
    )

    mixture_nll = -mixture_log_probability.mean()
    selected_expert_nll = -component_log_probabilities.gather(
        1, labels.unsqueeze(1)
    ).mean()
    classification = F.cross_entropy(
        logits,
        labels,
        label_smoothing=cfg.label_smoothing,
    )

    total = (
        mixture_nll
        + cfg.expert_alignment_weight * selected_expert_nll
        + cfg.regime_classification_weight * classification
    )
    metrics = {
        "total": float(total.detach().cpu()),
        "nll": float(mixture_nll.detach().cpu()),
        "accuracy": float(
            (logits.argmax(dim=1) == labels).float().mean().detach().cpu()
        ),
    }
    return total, metrics


def run_epoch(
    model: RegimeFlowMixture,
    loader: DataLoader,
    cfg: Config,
    optimiser: optim.Optimizer | None,
) -> dict[str, float]:
    training = optimiser is not None
    model.train(training)
    rows = []

    for context, target, labels in loader:
        context = context.to(DEVICE)
        target = target.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimiser.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            total, metrics = objective(model, context, target, labels, cfg)
            if training:
                total.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), cfg.gradient_clip
                )
                optimiser.step()
        rows.append(metrics)

    return {
        key: float(np.mean([row[key] for row in rows]))
        for key in rows[0]
    }


def fit_model(
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
) -> tuple[RegimeFlowMixture, dict[str, list[float]], int]:
    train_loader = make_loader(
        split, dataset, split.train_indices, cfg, shuffle=True
    )
    validation_loader = make_loader(
        split, dataset, split.validation_indices, cfg, shuffle=False
    )

    model = RegimeFlowMixture(split.context.shape[1], cfg).to(DEVICE)
    optimiser = optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimiser,
        mode="min",
        factor=0.5,
        patience=max(2, cfg.patience // 2),
    )

    history = {
        "train_loss": [],
        "validation_loss": [],
        "validation_nll": [],
        "validation_accuracy": [],
    }
    best_loss = float("inf")
    best_state = None
    best_epoch = 0
    stale_epochs = 0

    print("\nTraining...")
    for epoch in range(1, cfg.epochs + 1):
        train_metrics = run_epoch(model, train_loader, cfg, optimiser)
        with torch.no_grad():
            validation_metrics = run_epoch(
                model, validation_loader, cfg, optimiser=None
            )

        scheduler.step(validation_metrics["total"])
        history["train_loss"].append(train_metrics["total"])
        history["validation_loss"].append(validation_metrics["total"])
        history["validation_nll"].append(validation_metrics["nll"])
        history["validation_accuracy"].append(
            validation_metrics["accuracy"]
        )

        if validation_metrics["total"] < best_loss - 1e-4:
            best_loss = validation_metrics["total"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % cfg.print_every == 0:
            print(
                f"  Epoch {epoch:03d}/{cfg.epochs} | "
                f"train={train_metrics['total']:.4f} | "
                f"validation={validation_metrics['total']:.4f} | "
                f"regime_accuracy={validation_metrics['accuracy']:.3f}"
            )

        if stale_epochs >= cfg.patience:
            print(f"  Early stopping. Best epoch: {best_epoch}")
            break

    if best_state is None:
        raise RuntimeError("No valid model checkpoint was produced.")

    model.load_state_dict(best_state)
    return model, history, best_epoch

## Probabilistic prediction and evaluation

In [ ]:
def empirical_crps(samples: torch.Tensor, actual: torch.Tensor) -> torch.Tensor:
    sample_count = samples.shape[1]
    first_term = torch.mean(
        torch.abs(samples - actual.unsqueeze(1)), dim=1
    )
    sorted_samples, _ = torch.sort(samples, dim=1)
    ranks = torch.arange(
        1,
        sample_count + 1,
        device=samples.device,
        dtype=samples.dtype,
    )
    coefficients = 2.0 * ranks - sample_count - 1.0
    second_term = (
        sorted_samples * coefficients.unsqueeze(0)
    ).sum(dim=1) / (sample_count**2)
    return first_term - second_term


@torch.no_grad()
def predict(
    model: RegimeFlowMixture,
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
) -> pd.DataFrame:
    model.eval()
    outputs = []

    print("\nGenerating test forecasts...")
    indices = split.test_indices
    for start in range(0, len(indices), cfg.prediction_batch_size):
        batch_indices = indices[start:start + cfg.prediction_batch_size]
        context = torch.from_numpy(split.context[batch_indices]).float().to(DEVICE)
        target = torch.from_numpy(
            dataset.targets_log_vol[batch_indices]
        ).float().unsqueeze(1).to(DEVICE)

        log_samples, probabilities = model.sample_mixture(
            context, cfg.prediction_samples
        )
        volatility_samples = torch.exp(log_samples)
        actual = torch.exp(target.squeeze(1))

        mixture_log_probability, logits, _ = model.mixture_outputs(
            context, target
        )
        quantiles = torch.quantile(
            volatility_samples,
            torch.tensor(
                [0.05, 0.25, 0.50, 0.75, 0.95],
                device=DEVICE,
            ),
            dim=1,
        )

        batch = dataset.metadata.iloc[batch_indices].copy().reset_index(drop=True)
        batch["predicted_mean_vol"] = (
            volatility_samples.mean(dim=1).cpu().numpy()
        )
        batch["predicted_q05_vol"] = quantiles[0].cpu().numpy()
        batch["predicted_q25_vol"] = quantiles[1].cpu().numpy()
        batch["predicted_median_vol"] = quantiles[2].cpu().numpy()
        batch["predicted_q75_vol"] = quantiles[3].cpu().numpy()
        batch["predicted_q95_vol"] = quantiles[4].cpu().numpy()
        batch["negative_log_likelihood"] = (
            -mixture_log_probability
        ).cpu().numpy()
        batch["crps"] = empirical_crps(
            volatility_samples, actual
        ).cpu().numpy()
        batch["pit"] = (
            (volatility_samples <= actual.unsqueeze(1))
            .float()
            .mean(dim=1)
            .cpu()
            .numpy()
        )
        predicted_regime = logits.argmax(dim=1).cpu().numpy()
        batch["predicted_regime"] = predicted_regime
        batch["predicted_regime_name"] = REGIME_NAMES[predicted_regime]

        for regime_id, regime_name in enumerate(REGIME_NAMES):
            batch[f"probability_{regime_name.lower()}"] = (
                probabilities[:, regime_id].cpu().numpy()
            )
        outputs.append(batch)

    predictions = pd.concat(outputs, ignore_index=True)
    print(f"  Produced {len(predictions):,} test forecasts")
    return predictions


def qlike(actual: np.ndarray, predicted: np.ndarray) -> float:
    actual_variance = np.maximum(actual**2, EPS)
    predicted_variance = np.maximum(predicted**2, EPS)
    ratio = actual_variance / predicted_variance
    return float(np.mean(ratio - np.log(ratio) - 1.0))


def point_metrics(actual: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    error = predicted - actual
    correlation = (
        float(np.corrcoef(actual, predicted)[0, 1])
        if np.std(actual) > EPS and np.std(predicted) > EPS
        else float("nan")
    )
    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error**2))),
        "correlation": correlation,
        "qlike": qlike(actual, predicted),
    }


def confusion_matrix_numpy(
    actual: np.ndarray,
    predicted: np.ndarray,
    classes: int,
) -> np.ndarray:
    matrix = np.zeros((classes, classes), dtype=int)
    for true_value, predicted_value in zip(actual, predicted):
        matrix[int(true_value), int(predicted_value)] += 1
    return matrix


def evaluate(
    predictions: pd.DataFrame,
    cfg: Config,
) -> tuple[pd.DataFrame, np.ndarray]:
    actual = predictions["actual_vol"].to_numpy(dtype=float)
    model_prediction = predictions["predicted_median_vol"].to_numpy(dtype=float)

    model_row = {"model": "SigFlow mixture"}
    model_row.update(point_metrics(actual, model_prediction))
    model_row.update({
        "negative_log_likelihood": float(
            predictions["negative_log_likelihood"].mean()
        ),
        "crps": float(predictions["crps"].mean()),
        "coverage_50": float(
            (
                (actual >= predictions["predicted_q25_vol"].to_numpy())
                & (actual <= predictions["predicted_q75_vol"].to_numpy())
            ).mean()
        ),
        "coverage_90": float(
            (
                (actual >= predictions["predicted_q05_vol"].to_numpy())
                & (actual <= predictions["predicted_q95_vol"].to_numpy())
            ).mean()
        ),
    })

    true_regime = predictions["regime"].to_numpy(dtype=int)
    predicted_regime = predictions["predicted_regime"].to_numpy(dtype=int)
    model_row["regime_accuracy"] = float(
        np.mean(true_regime == predicted_regime)
    )

    rows = [model_row]
    for name, column in [
        ("Rolling volatility", "rolling_vol_baseline"),
        ("EWMA volatility", "ewma_vol_baseline"),
    ]:
        row = {"model": name}
        row.update(
            point_metrics(
                actual,
                predictions[column].to_numpy(dtype=float),
            )
        )
        rows.append(row)

    matrix = confusion_matrix_numpy(
        true_regime, predicted_regime, cfg.regimes
    )
    return pd.DataFrame(rows).set_index("model"), matrix

## Plots and saving

In [ ]:
def plot_results(
    history: dict[str, list[float]],
    predictions: pd.DataFrame,
    confusion: np.ndarray,
    cfg: Config,
) -> None:
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(10, 4))
    plt.plot(epochs, history["train_loss"], label="Training")
    plt.plot(epochs, history["validation_loss"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Objective")
    plt.title("Training history")
    plt.legend()
    plt.tight_layout()
    plt.show()

    for ticker in cfg.tickers:
        data = (
            predictions[predictions["ticker"] == ticker]
            .sort_values("origin_date")
            .copy()
        )
        if data.empty:
            continue

        dates = pd.to_datetime(data["origin_date"])
        plt.figure(figsize=(13, 5))
        plt.plot(dates, data["actual_vol"], label="Actual future volatility")
        plt.plot(
            dates,
            data["predicted_median_vol"],
            label="Predicted median",
        )
        plt.plot(
            dates,
            data["ewma_vol_baseline"],
            label="EWMA baseline",
            alpha=0.8,
        )
        plt.fill_between(
            dates,
            data["predicted_q05_vol"],
            data["predicted_q95_vol"],
            alpha=0.2,
            label="90% predictive interval",
        )
        plt.title(f"{ticker}: {cfg.horizon}-day volatility forecast")
        plt.ylabel("Annualised volatility")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(13, 3.5))
        plt.stackplot(
            dates,
            data["probability_low"],
            data["probability_medium"],
            data["probability_high"],
            labels=REGIME_NAMES,
            alpha=0.8,
        )
        plt.ylim(0, 1)
        plt.title(f"{ticker}: regime probabilities")
        plt.ylabel("Probability")
        plt.legend(loc="upper left", ncol=3)
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(7, 4))
    plt.hist(predictions["pit"], bins=np.linspace(0, 1, 11))
    plt.axhline(len(predictions) / 10, linestyle="--")
    plt.title("PIT calibration")
    plt.xlabel("PIT")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(5, 4))
    plt.imshow(confusion)
    plt.xticks(np.arange(3), REGIME_NAMES)
    plt.yticks(np.arange(3), REGIME_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Regime confusion matrix")
    for row in range(3):
        for column in range(3):
            plt.text(
                column,
                row,
                str(confusion[row, column]),
                ha="center",
                va="center",
            )
    plt.colorbar()
    plt.tight_layout()
    plt.show()


def save_results(
    model: RegimeFlowMixture,
    history: dict[str, list[float]],
    metrics: pd.DataFrame,
    predictions: pd.DataFrame,
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
) -> None:
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    predictions.to_csv(output_dir / "test_predictions.csv", index=False)
    metrics.to_csv(output_dir / "test_metrics.csv")
    pd.DataFrame(history).to_csv(
        output_dir / "training_history.csv", index=False
    )
    torch.save({
        "model_state_dict": model.state_dict(),
        "config": asdict(cfg),
        "feature_names": dataset.feature_names,
        "data_source": dataset.data_source,
        "feature_mean": split.standardiser.mean,
        "feature_scale": split.standardiser.scale,
        "regime_thresholds": split.regime_thresholds,
    }, output_dir / "sigflow_model.pt")

    with open(output_dir / "run_config.json", "w", encoding="utf-8") as file:
        json.dump(
            {**asdict(cfg), "actual_data_source": dataset.data_source},
            file,
            indent=2,
        )
    print("\nSaved outputs to:", output_dir.resolve())

# Run everything

This cell prints at every major stage. If the notebook stops, the last printed stage identifies the problem.

In [ ]:
def main(cfg: Config = CFG) -> dict[str, object]:
    print("=" * 72)
    print("Starting SigFlow-Sim")
    print("Mode:", "QUICK" if cfg.quick_mode else "FULL")
    print("=" * 72)

    set_seed(cfg.seed)
    dataset = build_market_dataset(cfg)
    split = prepare_split(dataset, cfg)
    model, history, best_epoch = fit_model(dataset, split, cfg)
    print("\nBest validation epoch:", best_epoch)

    predictions = predict(model, dataset, split, cfg)
    metrics, confusion = evaluate(predictions, cfg)

    print("\nEvaluation metrics")
    print(metrics.round(5).to_string())
    print("\nActual data source:", dataset.data_source)
    if not dataset.data_source.startswith("real_"):
        print(
            "WARNING: these are synthetic pipeline-test results, "
            "not claims about real stocks."
        )

    save_results(
        model,
        history,
        metrics,
        predictions,
        dataset,
        split,
        cfg,
    )
    plot_results(history, predictions, confusion, cfg)

    print("\nComplete.")
    return {
        "dataset": dataset,
        "split": split,
        "model": model,
        "history": history,
        "predictions": predictions,
        "metrics": metrics,
        "confusion_matrix": confusion,
    }


RESULTS = main(CFG)